
This enables the easy addition and change of the dictionary of filters.

'filter_data.js' is a js file that sets the value of filterData in the JustVertFilterApp.jsx which drives generation of the filter navbar.

the js file is essentially just a large dictionary with a first line that sets up the variable and a last line that exports it.

This script extracts the dictionary converts it to a tree structure enabling the addition of new children nodes before converting it back again.

In this example it is used to add in the TCGA codes and also a description. It will be needed again to update the ICD-O codes and to add in the SNOMED-CT codes.

In [35]:
import json
import os

In [2]:
from typing import Dict, Any, List, Optional

class TreeNode:
    """
    Represents a single node in the hierarchical tree structure.
    """
    def __init__(
        self,
        id: str,
        label: str,
        category: str,
        primary_group: str,
        count: str,
        description: Optional[str] = None,
        children: Optional[List['TreeNode']] = None,
    ):
        """
        Initializes a new tree node with data attributes.

        Args:
            id (str): The unique identifier for the node.
            label (str): The human-readable label or name.
            category (str): The category of the node.
            primary_group (str): The primary group classification.
            count (str): A count associated with the node.
            children (Optional[List['TreeNode']]): A list of child TreeNode objects.
        """
        self.id = id
        self.label = label
        self.category = category
        self.primary_group = primary_group
        self.count = count
        self.description = description
        self.children: List['TreeNode'] = children if children is not None else []

    def __repr__(self):
        """
        Provides a string representation for debugging.
        """
        return f"TreeNode(id='{self.id}', label='{self.label}', children={len(self.children)})"

        

    def print_tree(self, level=0):
        """
        Recursively prints the structure of the tree.
        """
        indent = "  " * level
        print(f"{indent}- {self.label} (ID: {self.id})")
        for child in self.children:
            child.print_tree(level + 1)


def build_tree(data: Dict[str, Any]) -> TreeNode:
    """
    Recursively builds the TreeNode structure from the raw dictionary data.

    Args:
        data (Dict[str, Any]): The raw dictionary data for a single node.

    Returns:
        TreeNode: The resulting object-oriented tree node.
    """
    # Extract core attributes
    node_id = data.get('id', '')
    label = data.get('label', '')
    category = data.get('category', '')
    primary_group = data.get('primaryGroup', '')
    count = data.get('count', '')
    description = data.get('description','')

    # Initialize the current node
    current_node = TreeNode(
        id=node_id,
        label=label,
        category=category,
        primary_group=primary_group,
        count=count,
        description=description,
    )

    # Check for children and recurse
    raw_children = data.get('children', {})
    if raw_children and isinstance(raw_children, dict):
        # Iterate over the values (the child dictionaries)
        for child_dict in raw_children.values():
            # Recursively build the child node
            child_node = build_tree(child_dict)
            current_node.children.append(child_node)

    return current_node

In [24]:
def load_dictionary(filter_path="filter_data.js"):
    with open(filter_path) as f:
        variable_setting, dict_text, export_statement =  f.read().split("\n")
        return variable_setting, json.loads(dict_text[:-1]), export_statement

In [27]:
def build_dict(tree: TreeNode) -> Dict:
    dictionary = {}
    dictionary["id"] = tree.id
    dictionary["label"] = tree.label
    dictionary["category"] = tree.category
    dictionary["primary_group"] = tree.primary_group
    dictionary["count"] = tree.count
    dictionary["description"] = tree.description
    if tree.children:
        dictionary["children"] = {}
        for child in tree.children:
            dictionary["children"][child.id] = build_dict(child)
    return dictionary

In [32]:
def add_tcga(output_path, tcga_path="TCGA_codes.txt", filter_path="filter_data.js"):
    # get the info for the tcga codes
    with open(tcga_path) as f:
        codes = [i.split("\t") for i in f.read().split("\n")]
        
    # make the new child tcga
    node = TreeNode(
            id='0_0_4',
            label='tcga',
            category='cancerTypes',
            primary_group='cancer-type',
            count=0,
            description="")
    # pull out the ids, labels and descriptions from the list and make new children for tcga 
    for i,code in enumerate(sorted(codes)):
        childNode = TreeNode(
            id=f"0_0_4_{str(i)}",
            label=code[0],
            category='cancerTypes',
            primary_group='cancer-type',
            count=0,
            description=code[1])
        node.children.append(childNode)
        
    variable_setting, filter_dictionary, export_statement = load_dictionary()
    # Turn the dictionary into a tree add the node and turn it back into a dictionary
    tree = build_tree(filter_dictionary["0_0"])
    tree.children.append(node)
    new_filter_dict = {"0_0" : build_dict(tree), 
                       "0_1": filter_dictionary["0_1"],
                       "0_2": filter_dictionary["0_2"]}
    amended_text = f"""{variable_setting}\n{new_filter_dict};\n{export_statement}"""
    with open(output_path, "w") as f:
        f.write(amended_text)

In [33]:
add_tcga("new_filter_data.js")